In [37]:
import json
from student.agent.agent_raspa import RaspaAgent

ag = RaspaAgent(path=".")

K_TO_KJMOL = 1.380649e-23 * 6.02214076e23 / 1000  # Convert K to kJ/mol
RT = 300 * 8.31 / 1000 # in kJ/mol

A CSD path is required to access the coremof files.
Logger: Created new log file at /Users/henrikseng/Desktop/StudentAgent/benchmarking/manual_solutions/single/ads_dil___l/logs/agent_2025-12-02 16-56-08-9d59.json


In [2]:
task = "ads_dil___l"

# Copy task
task_description = json.load(open("../../../input/tasks_single.json"))[task][0]
with open("task.txt", "w") as f:
    f.write(task_description)
print(task_description)

Determine the adsorption enthalpy of n-hexane on IRMOF-13 using a simulation at infinite dilution at 300 K given the helium void fraction of 0.8


In [21]:
# Load framework
ag.tools["framework_loader"].run("IRMOF-19")

# Load molecule
ag.tools["molecule_loader"].run("n-hexane")

# Create input file
input_content = """SimulationType                MonteCarlo
NumberOfCycles                100000
NumberOfInitializationCycles  10000
PrintEvery                    5000

Forcefield                    local               
CutOff                        14.0                
RemoveAtomNumberCodeFromLabel yes

Framework 0
FrameworkName framework
UnitCells 2 2 2
ExternalTemperature 300
ExternalPressure 1e5

Component 0 MoleculeName             n-hexane
            MoleculeDefinition       local
            IdealGasRosenbluthWeight 0.0029442
            TranslationProbability   1.0
            ReinsertionProbability   1.0
            PartialReinsertionProbability 1.0
            RotationProbability      1.0
            CreateNumberOfMolecules  1

"""
ag.tools["input_file"].run(input_content)

CIF Name: ./simulation_2/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./simulation_2/framework_pacman.cif
RASPA UnitCells: 2 2 2
Loading unknown molecule:  n-hexane


'<tool response name=input_file>\nSuccessfully generated: <file name=./simulation_2/simulation.input></file>\n</tool response>'

In [22]:
ag.tools["execute_raspa"].run()

"<tool response name=execute_raspa>\n<terminal_output>('', '_cell_length_a: 17.146900\\n_cell_length_b: 23.322200\\n_cell_length_c: 25.255200\\n_cell_length_alpha: 90.000000\\n_cell_length_beta: 90.000000\\n_cell_length_gamma: 90.000000\\n_symmetry_space_group_name_Hall: P 1 found space group: 1\\n_symmetry_space_group_name_H-M: P 1 found space group: 1\\n_symmetry_Int_Tables_number: 1\\nspace group found from symmetry elements: 1 (nr elements: 1)\\nEnd reading cif-file\\n')</terminal_output>\\n (IMPORTANT: new, empty working directory created! To rerun, you must create all input files again!)\n</tool response>"

In [27]:
print(ag.tools["output_parser"].run("simulation_2/Output/System_0/output_framework_2.2.2_300.000000_100000.data", "Total energy, tail correction and internal energies (rotation, vibrations, ...) - all with uncertainties if provided"))

<tool response name=output_parser>
- Total energy [K]: 539464233.93794 +/- 180.07662
- Tail-correction energy [K]: -11013.10531 +/- 0.00017
- Internal energies (rotation, vibrations, ...):
  - Average Adsorbate Bend angle energy [K]: 539481297.89929 +/- 11.104
  - Average Adsorbate Torsion energy [K]: 843.41488 +/- 7.56309
  - Average Adsorbate Intra Van der Waals energy [K]: -35.26831 +/- 1e-05
</tool response>


In [24]:
# The enthalpy of adsorption can be calculated from the output data using:
# ∆H= ⟨Uhg⟩−⟨Uh⟩−⟨Ug⟩−RT - tail corrections (since only 1 molecule!)
# 
# R is the universal gas constant, and T is the temperature in Kelvin.
# RT is the enthalpy per particle of the ideal bulk phase. 
# It accounts for the work to push the gas adsorbates into the fluid phase when it desorbs.
# 
# ⟨Uhg⟩ = average total energy of framework + adsorbate system
# ⟨Uh⟩ = average total energy of the empty framework (0 if rigid)
# ⟨Ug⟩ = average total energy of the adsorbate in the gas phase

In [ ]:
tail_correction = -11013.10531
ug = 539482112  # from separate simulation of n-hexane in gas phase

raw_enthalpy = 	539464233.93794 # 2489.674292 [K]
enthalpy = raw_enthalpy - tail_correction -ug - RT/K_TO_KJMOL

In [41]:
print(f"Enthalpy in kJ/mol = {enthalpy * K_TO_KJMOL}")

Enthalpy in kJ/mol = -57.08091927340715
